# Notebook 11 - Hybrid Text Plus Vision Retrieval

This notebook combines text retrieval and visual retrieval into one ranking so evidence can come from either modality without forcing a false choice.


## What you will learn

- how to fuse text and vision retrieval scores
- how to inspect the impact of different weighting schemes
- how to preserve evidence provenance across retrieval channels


In [ ]:
!pip install -q numpy pandas matplotlib
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "11_hybrid_text_plus_vision_retrieval"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Build a fused ranking table

Hybrid retrieval becomes tangible once both channels live in the same table.


In [ ]:
results = pd.DataFrame([
    {"candidate": "page-1", "text_score": 0.82, "vision_score": 0.61},
    {"candidate": "page-2", "text_score": 0.44, "vision_score": 0.93},
    {"candidate": "page-3", "text_score": 0.77, "vision_score": 0.54},
])
results


## Step 2: Calibrate the fusion weight

The weight should reflect where the evidence lives. Document pages often benefit from a stronger visual component than ordinary text chunks.


In [ ]:
def fuse_scores(text_score, vision_score, alpha=0.5):
    return alpha * text_score + (1 - alpha) * vision_score

for alpha in [0.2, 0.5, 0.8]:
    results[f"fused_{alpha}"] = results.apply(lambda row: fuse_scores(row.text_score, row.vision_score, alpha=alpha), axis=1)
results.round(3)


## Step 3: Keep evidence provenance

A good retrieval row records not just the score but why the candidate was retrieved.


In [ ]:
provenance = pd.DataFrame([
    {"candidate": "page-1", "top_text_hit": "invoice total", "top_visual_hit": "bottom-right total box"},
    {"candidate": "page-2", "top_text_hit": "weak OCR match", "top_visual_hit": "table title + row alignment"},
    {"candidate": "page-3", "top_text_hit": "policy section", "top_visual_hit": "chart legend"},
])
results.merge(provenance, on="candidate").sort_values("fused_0.5", ascending=False)


## Exercises and extensions

1. Search over alpha values and plot grounded_success against alpha.
1. Add a reranker score column and compare two-stage versus one-stage ranking.
